# CYGNSS L1 AZ — ISMN in-situ skill figures (fixed-operator arms, 2020–2022)

This notebook makes CYGNSS-paper-style in-situ skill figures (`projects/cygnss_da/notebooks/CYG_insitu_plotter_100325.ipynb`) for the
AZ limited-domain fixed-operator runs.

**Layout.** Rows are surface and root zone; columns are R, anomR and ubRMSD. The x-axis is the experiment. Series are the networks
plus all stations pooled.

**Figures produced**

1. Raw skill (mean over stations).
2. Δ vs OL (the control).
3. Δ vs L3-only, which is the direct L1-vs-L3 comparison.

**Station set.** By default, only stations on tiles that receive CYGNSS L1 obs: at least `L1_MIN_OBS` unfiltered L1 obs over 2020–2022.
About 78% of the ISMN stations in the GEOSldas box are on tiles with no L1 obs. See
`runs/cygl1_fixedop_ismn_validation_2020_2022.md`.

**Confidence intervals (95%)**

- Δ panels, `CI_METHOD="paired_t"` (default, same as the CYGNSS paper's `paired_delta_stats`): mean of the per-station paired
  differences ± t(0.975, n−1)·sd/√n.
- Δ panels, `CI_METHOD="tile_bootstrap"`: percentile interval from resampling model tiles with all their stations. Stations on one tile
  share one model time series, so this interval is more conservative. On the L1-area set it gives the same significance verdicts.
- Raw panels: t-interval on the station mean. The paper's raw-panel CI used per-station Fisher-z bounds, which this station CSV does not
  contain.
- ubRMSD Δ is plotted as **improvement** (reference − experiment), as in the paper, so for every metric up means better.

**Inputs** (running locally: `output/` is gitignored. Extract `cygl1_ismn_fixedop_2020_2022_inputs.tar.gz` inside
`projects/CYGNSS_L1_AZ/` first. Needs numpy, pandas, scipy and matplotlib; xarray is only needed on Discover.)


- `ismn_skill_stations.csv` from job 58621216 (`jobs/run_ismn_ol_da_skill_az.sbatch`).
- A per-tile L1 obs count. It is read from the L3-only arm's 2020–2022 temporal stats nc4 (species 12, unfiltered monitor stream), or
  from a cached CSV when the nc4 is not available (e.g. on a laptop).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t
from pathlib import Path

# ===============================
# CONFIG
# ===============================
# Locate the project from either the repository root or the notebook directory.
cwd = Path.cwd().resolve()
project_rel = Path("projects/CYGNSS_L1_AZ")
candidates = [cwd, cwd.parent, *(root / project_rel for root in (cwd, *cwd.parents)),
              Path("/discover/nobackup/projects/land_da/geosldas-analysis") / project_rel]
PROJ = next((path for path in candidates if (path / "notebooks").is_dir()), None)
if PROJ is None:
    raise FileNotFoundError(f"Could not locate projects/CYGNSS_L1_AZ from working directory {cwd}")
ISMN_DIR = PROJ / "output/ismn_fixedop_20200101_20221231"
STATIONS_CSV = ISMN_DIR / "ismn_skill_stations.csv"
L1_COUNT_NC4 = PROJ / "output/postproc_paired_density/stats_output/temporal_stats_DA_paired_l3_fixedop_20200101_20221231.nc4"
L1_COUNT_CSV = ISMN_DIR / "tile_l1_obs_count_2020_2022.csv"   # cache written below; used if the nc4 is not available
OUT_DIR = PROJ / "output/ismn_fixedop_figures"

L1_MIN_OBS = 100            # station set: tiles with >= this many unfiltered L1 obs 2020-2022; set to None for all stations
CI_METHOD = "paired_t"      # Δ panels: "paired_t" (CYGNSS paper) or "tile_bootstrap"
STYLE = "dots"              # "dots" (CYGNSS paper) or "bars"
N_BOOT = 5000

# Runs as named in the station CSV -> x-axis labels (order = x-axis)
experiment_names = ["OL", "L1coh", "L1coh_err39", "L3", "SMAP"]
expt_labels      = ["OL", "L1", "L1 err3.9", "L3", "SMAP"]

# Series: networks present in the L1 area, plus all stations pooled ("All")
networks = ["All", "SCAN", "USCRN", "SNOTEL", "SOILSCAPE"]
palette = {"All": "#000000", "SCAN": "#0072B2", "USCRN": "#E69F00", "SNOTEL": "#56B4E9", "SOILSCAPE": "#009E73"}
markers = {"All": "D", "SCAN": "o", "USCRN": "s", "SNOTEL": "v", "SOILSCAPE": "^"}

# Plot look
label_fs, tick_fs, legend_fs = 11, 10, 12
dot_size, capsize = 55, 2

panel_info_means = [
    ("R",      r"$R$ (-)",                None),
    ("anomR",  r"anomR (-)",              None),
    ("ubRMSE", r"ubRMSD ($m^3\,m^{-3}$)", None),
]
def panel_info_delta(ref_label):
    return [
        ("R",      rf"Δ$R$ (EXP − {ref_label})",           None),
        ("anomR",  rf"ΔanomR (EXP − {ref_label})",         None),
        ("ubRMSE", rf"ΔubRMSD ({ref_label} − EXP)",        None),
    ]
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ===============================
# LOAD STATIONS + L1 COVERAGE MASK
# ===============================
if not STATIONS_CSV.exists():
    raise FileNotFoundError(
        f"{STATIONS_CSV} not found. output/ is gitignored: to run locally, extract "
        "cygl1_ismn_fixedop_2020_2022_inputs.tar.gz (from /gpfsm/dnb06/projects/p284/cygl1_operator_test/) "
        "inside projects/CYGNSS_L1_AZ/ so the files land in output/ismn_fixedop_20200101_20221231/.")
st = pd.read_csv(STATIONS_CSV)

if L1_COUNT_NC4.exists():
    import xarray as xr
    n_l1 = xr.open_dataset(L1_COUNT_NC4)["N_data"].values[:, 12].clip(0)   # species 12 = CYGNSS_L1
    pd.DataFrame({"tile_index": np.arange(n_l1.size), "n_l1_obs": n_l1}).to_csv(L1_COUNT_CSV, index=False)
else:
    n_l1 = pd.read_csv(L1_COUNT_CSV).sort_values("tile_index")["n_l1_obs"].to_numpy()
st["n_l1_obs"] = n_l1[st["tile_index"].to_numpy()]

if L1_MIN_OBS is not None:
    st = st[st["n_l1_obs"] >= L1_MIN_OBS].copy()
    set_label = f"L1 area (tiles with ≥{L1_MIN_OBS} L1 obs)"
else:
    set_label = "all stations in GEOSldas box"
st = st[st["run"].isin(experiment_names)]

ol = st[st["run"] == "OL"]
print(set_label)
for dom in ["surface", "rz"]:
    x = ol[ol["domain"] == dom]
    print(f"  {dom}: {x.station_key.nunique()} stations on {x.tile_index.nunique()} tiles;",
          x.network.value_counts().to_dict())

In [ ]:
# ===============================
# STATS
# ===============================
def paired_delta_stats(arr_cntl, arr_exp, up_is_better=True, alpha=0.05):
    """Paired per-site differences and t-interval (as in CYG_insitu_plotter_100325.ipynb).
    up_is_better=True -> Δ = EXP - CNTL; False -> Δ = CNTL - EXP (ubRMSE)."""
    sign = +1 if up_is_better else -1
    di = sign * (np.asarray(arr_exp, float) - np.asarray(arr_cntl, float))
    di = di[np.isfinite(di)]
    n = di.size
    if n == 0:
        return np.nan, np.nan, 0
    sd = np.std(di, ddof=1) if n > 1 else 0.0
    k = t.ppf(1 - alpha / 2, df=max(n - 1, 1))
    return di.mean(), k * sd / np.sqrt(max(n, 1)), n

_rng = np.random.default_rng(0)
def tile_bootstrap_delta(d_vals, tiles, n_boot=N_BOOT):
    """Percentile CI of the station-mean Δ, resampling tiles (with all their stations). Returns (mean, lo_mag, up_mag, n)."""
    ok = np.isfinite(d_vals)
    d_vals, tiles = d_vals[ok], tiles[ok]
    if d_vals.size == 0:
        return np.nan, np.nan, np.nan, 0
    g = pd.Series(d_vals).groupby(tiles).agg(["sum", "count"])
    S, C = g["sum"].to_numpy(), g["count"].to_numpy()
    idx = _rng.integers(0, len(g), (n_boot, len(g)))
    bs = S[idx].sum(1) / C[idx].sum(1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    m = d_vals.mean()
    return m, m - lo, hi - m, d_vals.size

def mean_t_ci(v, alpha=0.05):
    v = np.asarray(v, float); v = v[np.isfinite(v)]; n = v.size
    if n < 2:
        return (v.mean() if n else np.nan), np.nan, n
    return v.mean(), t.ppf(1 - alpha / 2, n - 1) * v.std(ddof=1) / np.sqrt(n), n

metrics = ["R", "anomR", "ubRMSE"]
depths = ["surface", "rz"]

def station_table(dom, net, metric):
    x = st[st["domain"] == dom]
    if net != "All":
        x = x[x["network"] == net]
    return x.pivot_table(index=["tile_index", "station_key"], columns="run", values=metric).reset_index()

def build_raw():
    shape = (len(networks), 2, len(experiment_names))
    means = {m: np.full(shape, np.nan) for m in metrics}
    cis = {m: np.full((len(networks), 2, 2, len(experiment_names)), np.nan) for m in metrics}
    Ns = {m: np.zeros(shape, int) for m in metrics}
    for ni, net in enumerate(networks):
        for di, dom in enumerate(depths):
            for m in metrics:
                p = station_table(dom, net, m)
                for ei, ex in enumerate(experiment_names):
                    if ex not in p:
                        continue
                    mu, hw, n = mean_t_ci(p[ex])
                    means[m][ni, di, ei], Ns[m][ni, di, ei] = mu, n
                    cis[m][ni, :, di, ei] = hw
    return means, cis, Ns

def build_delta(ref="OL", method=CI_METHOD):
    shape = (len(networks), 2, len(experiment_names))
    means = {m: np.full(shape, np.nan) for m in metrics}
    cis = {m: np.full((len(networks), 2, 2, len(experiment_names)), np.nan) for m in metrics}
    Ns = {m: np.zeros(shape, int) for m in metrics}
    for ni, net in enumerate(networks):
        for di, dom in enumerate(depths):
            for m in metrics:
                up = m != "ubRMSE"
                p = station_table(dom, net, m)
                if ref not in p:
                    continue
                for ei, ex in enumerate(experiment_names):
                    if ex not in p:
                        continue
                    if ex == ref:
                        means[m][ni, di, ei], cis[m][ni, :, di, ei] = 0.0, 0.0
                        Ns[m][ni, di, ei] = p[ref].notna().sum()
                        continue
                    if method == "paired_t":
                        mu, hw, n = paired_delta_stats(p[ref], p[ex], up_is_better=up)
                        lo_mag = up_mag = hw
                    else:
                        d = (1 if up else -1) * (p[ex] - p[ref]).to_numpy(float)
                        mu, lo_mag, up_mag, n = tile_bootstrap_delta(d, p["tile_index"].to_numpy())
                    means[m][ni, di, ei], Ns[m][ni, di, ei] = mu, n
                    cis[m][ni, 0, di, ei], cis[m][ni, 1, di, ei] = lo_mag, up_mag
    return means, cis, Ns

means_raw, cis_raw, Ns_raw = build_raw()
delta_ol = build_delta("OL")
delta_l3 = build_delta("L3")

In [ ]:
# ===============================
# PLOT (CYGNSS-paper layout)
# ===============================
def plot_overlay(means_dict, cis_dict, Ns_dict, panel_info, outfile, expt_idx=None, zero_line=False, suptitle=None):
    """means_dict[m]: [net, depth, expt]; cis_dict[m]: [net, 2(lo,up), depth, expt] (magnitudes); Ns_dict[m]: [net, depth, expt]"""
    expt_idx = np.arange(len(expt_labels)) if expt_idx is None else np.asarray(expt_idx)
    x_tick_labels = [expt_labels[i] for i in expt_idx]
    fig = plt.figure(figsize=(12.5, 7.8), constrained_layout=True)
    axs = fig.subplot_mosaic([["legend"] * 3, ["s0", "s1", "s2"], ["r0", "r1", "r2"]],
                             gridspec_kw={"height_ratios": [0.14, 1.0, 1.0]})
    for lab, key in zip(["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"], ["s0", "s1", "s2", "r0", "r1", "r2"]):
        axs[key].text(0.02, 0.97, lab, transform=axs[key].transAxes, fontsize=12, va="top", ha="left")
    ax_leg = axs["legend"]; ax_leg.axis("off")
    if suptitle:
        ax_leg.set_title(suptitle, fontsize=11)

    x = np.arange(len(expt_idx))
    nn = len(networks)
    width = 0.8 / nn
    offsets = (np.arange(nn) - (nn - 1) / 2) * (width if STYLE == "bars" else 0.09)

    for col, (mk, ylab, ylim) in enumerate(panel_info):
        for depth in [0, 1]:
            ax = axs[("s" if depth == 0 else "r") + str(col)]
            ax.grid(axis="y", color="lightgrey", zorder=0); ax.set_axisbelow(True)
            if zero_line:
                ax.axhline(0, color="0.3", lw=0.8, zorder=1)
            for ni, net in enumerate(networks):
                y = means_dict[mk][ni, depth, expt_idx]
                ci = np.asarray(cis_dict[mk][ni][:, depth][:, expt_idx], float)   # (2, n_expt)
                xpos = x + offsets[ni]
                lab = net if (depth == 0 and col == 0) else None
                if STYLE == "bars":
                    ax.bar(xpos, y, width=width, color=palette[net], alpha=0.85 if net != "All" else 0.6,
                           zorder=3, label=lab)
                else:
                    ax.scatter(xpos, y, s=dot_size, marker=markers[net], color=palette[net], edgecolor="none",
                               zorder=3, label=lab)
                ax.errorbar(xpos, y, yerr=ci, fmt="none", ecolor="gray", elinewidth=1.0, capsize=capsize, zorder=4)
            if ylim is not None:
                ax.set_ylim(*ylim)
            ax.set_xticks(x, x_tick_labels if depth == 1 else [""] * len(x), fontsize=tick_fs)
            ax.set_ylabel(f"{'Surface' if depth == 0 else 'Rootzone'} {ylab}", fontsize=label_fs)
            ntext = " | ".join(f"{networks[ni]} n={int(Ns_dict[mk][ni, depth, expt_idx[0]])}"
                               for ni in range(nn) if Ns_dict[mk][ni, depth, expt_idx[0]] > 0)
            ax.set_title(ntext, fontsize=8.5)
    handles, labels = axs["s0"].get_legend_handles_labels()
    ax_leg.legend(handles, labels, loc="center", ncols=len(labels), fontsize=legend_fs, frameon=False)
    fig.savefig(outfile, dpi=300, bbox_inches="tight", pad_inches=0.04)
    plt.show()

tag = f"l1min{L1_MIN_OBS}" if L1_MIN_OBS is not None else "allstations"
ci_note = {"paired_t": "paired t 95% CI", "tile_bootstrap": "tile-cluster bootstrap 95% CI"}[CI_METHOD]


In [ ]:
# FIGURE 1: raw skill
plot_overlay(means_raw, cis_raw, Ns_raw, panel_info_means, OUT_DIR / f"ismn_raw_skill_{tag}_{STYLE}.png",
             suptitle=f"{set_label}, 2020–2022 — mean over stations ± t 95% CI")

In [ ]:
# FIGURE 2: Δ vs OL (drop OL from the x-axis)
idx = [i for i, e in enumerate(experiment_names) if e != "OL"]
plot_overlay(*delta_ol, panel_info_delta("OL"), OUT_DIR / f"ismn_delta_vs_OL_{tag}_{CI_METHOD}_{STYLE}.png",
             expt_idx=idx, zero_line=True, suptitle=f"{set_label} — Δ vs OL, {ci_note}")

In [ ]:
# FIGURE 3: Δ vs L3-only (direct L1-vs-L3 comparison; positive = better than L3)
idx = [i for i, e in enumerate(experiment_names) if e not in ("L3", "OL")]
plot_overlay(*delta_l3, panel_info_delta("L3"), OUT_DIR / f"ismn_delta_vs_L3_{tag}_{CI_METHOD}_{STYLE}.png",
             expt_idx=idx, zero_line=True, suptitle=f"{set_label} — Δ vs L3-only, {ci_note}")

In [ ]:
# Summary table (All stations series) -> CSV
rows = []
for name, (mn, ci, N) in [("vs_OL", delta_ol), ("vs_L3", delta_l3)]:
    for m in metrics:
        for di, dom in enumerate(depths):
            for ei, ex in enumerate(experiment_names):
                v = mn[m][0, di, ei]
                if not np.isfinite(v) or (name == "vs_OL" and ex == "OL") or (name == "vs_L3" and ex == "L3"):
                    continue
                lo, up = ci[m][0, :, di, ei]
                rows.append(dict(comparison=name, metric=m, domain=dom, run=ex, delta=v, ci_lo=v - lo, ci_hi=v + up,
                                 significant=(v - lo > 0) or (v + up < 0), n=N[m][0, di, ei]))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / f"ismn_delta_summary_{tag}_{CI_METHOD}.csv", index=False)
summary.round(4)